In [ ]:
import json
import os
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, Image, clear_output


In [ ]:

def deduplicate_dataset(data_list):
    # not hashable, so we have to do this manually
    deduped = []
    for item in data_list:
        if item not in deduped:
            deduped.append(item)
    return deduped

# Load hypotheses from jsonl file
input_jsonl = "exported_hypotheses.jsonl"
input_jsonl = os.path.expanduser(input_jsonl)
with open(input_jsonl, "r") as f:
    data = [json.loads(line) for line in f]
    len_before = len(data)
    data = deduplicate_dataset(data)
    print(f"Removed {len_before - len(data)} duplicate hypotheses, now {len(data)} unique entries.")


In [ ]:
images_base_path = Path("VisDiffBench").expanduser()

In [ ]:
def load_jsonls():
    jsons = []
    for f in ["easy.jsonl", "medium.jsonl", "hard.jsonl"]:
        with open(images_base_path / f, "r") as file:
            for line in file:
                jsons.append(json.loads(line))
    return jsons

def get_img_paths(set1_name, set2_name, jsons):
    for entry in jsons:
        if entry["set1"] == set1_name and entry["set2"] == set2_name:
            return [images_base_path / p for p in entry["set1_images"]], [images_base_path / p for p in entry["set2_images"]]
    raise ValueError(f"Could not find image paths for sets {set1_name} and {set2_name}")


In [ ]:
jsons = load_jsonls()

In [ ]:

if "manual_label" in data[0]:
    labels = [example.get("manual_label", None) for example in data]
else:
    labels = [None] * len(data)
current_idx = labels.index(None) if None in labels else 0

def show_example(idx):
    clear_output(wait=True)
    example = data[idx]
    set1_images, set2_images = get_img_paths(example['group_a'], example['group_b'], jsons)
    print(f"{set1_images=}, {set2_images=}")
    img_widgets1 = [widgets.Image.from_file(filename=str(img_path), width=200) for i, img_path in enumerate(set1_images) if i < 20]
    img_widgets2 = [widgets.Image.from_file(filename=str(img_path), width=200) for i, img_path in enumerate(set2_images) if i < 20]
    display(widgets.HBox(img_widgets1))
    display(widgets.HBox(img_widgets2))
    display(widgets.HBox([prev_button, next_button, export_button]))
    display(label_widget)

    text = f"<b>Hypothesis:</b> {example['hypothesis']}<br><b>Ground Truth:</b> {example['group_a']} -----> {example['group_b']}<br><b>Current Label:</b> {labels[idx]}<br><b>Example {idx+1}/{len(data)}</b>"
    display(widgets.HTML(value=text))


def on_label_change(change):
    global current_idx
    labels[current_idx] = change['new'] if change['new'] != -1 else None
    #show_example(current_idx)

def on_next_clicked(b):
    global current_idx
    if current_idx < len(data) - 1:
        current_idx += 1
        label_widget.value = labels[current_idx] if labels[current_idx] is not None else -1
        show_example(current_idx)

def on_prev_clicked(b):
    global current_idx
    if current_idx > 0:
        current_idx -= 1
        label_widget.value = labels[current_idx] if labels[current_idx] is not None else -1
        show_example(current_idx)

def on_export_clicked(b):
    output_jsonl = "labeled_hypotheses.jsonl"
    with open(output_jsonl, "w") as f:
        for example, label in zip(data, labels):
            example['manual_label'] = label
            f.write(json.dumps(example) + "\n")
    print(f"Exported labeled data to {output_jsonl}")


In [ ]:

label_widget = widgets.BoundedIntText(
    value=-1,
    min=-1,
    max=2,
    step=1,
    description='Label:',
    disabled=False
)
label_widget.observe(on_label_change, names='value')
next_button = widgets.Button(description="Next")
prev_button = widgets.Button(description="Previous")
export_button = widgets.Button(description="Export")

next_button.on_click(on_next_clicked)
prev_button.on_click(on_prev_clicked)
export_button.on_click(on_export_clicked)

display(label_widget, prev_button, next_button, export_button)
show_example(current_idx)